# Week 2, Day 2 — Does Memory Help on ABUK?

Yesterday, an MLP predicted **ABUK's next-session return** from one row of market features. Today we keep the same stock, features, chronological 70/30 split, target, and evaluation—but give an LSTM the last **5 feature rows**.

**Question:** does seeing a short history help the LSTM beat the MLP on unseen future returns?

In [ ]:
import os, sys
while not os.path.isdir('src') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
sys.path.insert(0, 'src')

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from tradinglab.data_feed import DataFeed
from tradinglab.features import FEATURE_NAMES, feature_columns
from tradinglab.ml import train_model, predict
from tradinglab.models import DeepMLP, LSTMRegressor

torch.set_num_threads(1)
np.random.seed(7); torch.manual_seed(7)
plt.style.use('seaborn-v0_8-darkgrid')

## 1. Same ABUK problem, now with a five-day window

Features at date `t` predict the return from `t` to `t+1`. We split through time first, learn normalization from the training period only, and build five-day sequences separately inside train and test so no window crosses the boundary.

In [ ]:
SYMBOL = 'ABUK'
SEQ_LEN = 5
feed = DataFeed.from_dir('data/egx')
asset = feed.symbols.index(SYMBOL)

X_full = feature_columns(feed, asset)
y_full = np.full(feed.n_days, np.nan)
y_full[:-1] = feed.returns[1:, asset]
valid = ~np.isnan(X_full).any(axis=1) & ~np.isnan(y_full)
X = X_full[valid].astype('float32')
y = y_full[valid].astype('float32')
dates = feed.dates[valid]

split = int(0.70 * len(X))
X_train_raw, X_test_raw = X[:split], X[split:]
y_train_raw, y_test_raw = y[:split], y[split:]
dates_train_raw, dates_test_raw = dates[:split], dates[split:]

x_mean, x_std = X_train_raw.mean(0), X_train_raw.std(0)
x_std[x_std < 1e-8] = 1.0
X_train_scaled = ((X_train_raw - x_mean) / x_std).astype('float32')
X_test_scaled = ((X_test_raw - x_mean) / x_std).astype('float32')
y_mean, y_std = float(y_train_raw.mean()), float(y_train_raw.std())
y_std = y_std if y_std >= 1e-8 else 1.0
y_train_scaled = ((y_train_raw - y_mean) / y_std).astype('float32')
y_test_scaled = ((y_test_raw - y_mean) / y_std).astype('float32')

def make_sequences(X_part, y_part, dates_part, length=5):
    X_seq = np.array([X_part[i-length+1:i+1] for i in range(length-1, len(X_part))], dtype='float32')
    return X_seq, y_part[length-1:], dates_part[length-1:]

Xtr_seq, ytr, dates_train = make_sequences(X_train_scaled, y_train_scaled, dates_train_raw, SEQ_LEN)
Xte_seq, yte, dates_test = make_sequences(X_test_scaled, y_test_scaled, dates_test_raw, SEQ_LEN)
y_train_actual = y_train_raw[SEQ_LEN-1:]
y_test_actual = y_test_raw[SEQ_LEN-1:]

# Same targets and dates: MLP sees today; LSTM sees the last five days.
Xtr_mlp, Xte_mlp = Xtr_seq[:, -1, :], Xte_seq[:, -1, :]

print(f'{SYMBOL}: {len(X):,} valid rows | {len(FEATURE_NAMES)} features')
print('Features:', FEATURE_NAMES)
print(f'Train sequences: {len(Xtr_seq):,} | Test sequences: {len(Xte_seq):,}')
print(f'Each LSTM input has shape: {SEQ_LEN} days × {Xtr_seq.shape[2]} features')
print('Chronology preserved:', dates_train[-1] < dates_test[0])

## 2. Train the basic LSTM and yesterday's MLP

The LSTM matches the course example: one recurrent layer with 32 hidden units. The MLP uses yesterday's baseline architecture. Both use the same seed, epochs, learning rate, targets, and dates.

In [ ]:
EPOCHS = 200
LR = 1e-3

torch.manual_seed(7)
lstm = LSTMRegressor(n_features=Xtr_seq.shape[2], hidden=32)
lstm_history = train_model(lstm, Xtr_seq, ytr, Xte_seq, yte, epochs=EPOCHS, lr=LR)

torch.manual_seed(7)
mlp = DeepMLP(n_features=Xtr_mlp.shape[1], hidden=32, n_hidden_layers=2)
mlp_history = train_model(mlp, Xtr_mlp, ytr, Xte_mlp, yte, epochs=EPOCHS, lr=LR)

lstm_train = predict(lstm, Xtr_seq) * y_std + y_mean
lstm_test = predict(lstm, Xte_seq) * y_std + y_mean
mlp_train = predict(mlp, Xtr_mlp) * y_std + y_mean
mlp_test = predict(mlp, Xte_mlp) * y_std + y_mean

print(f'LSTM parameters: {sum(p.numel() for p in lstm.parameters()):,}')
print(f'MLP parameters:  {sum(p.numel() for p in mlp.parameters()):,}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
epochs = np.arange(1, EPOCHS + 1)

axes[0].plot(epochs, lstm_history['train'], label='Training loss')
axes[0].plot(epochs, lstm_history['test'], label='Testing loss')
axes[0].set(title='LSTM loss curves', xlabel='Epoch', ylabel='Standardized-return MSE')
axes[0].legend()

axes[1].plot(epochs, lstm_history['test'], label='LSTM test loss')
axes[1].plot(epochs, mlp_history['test'], label='MLP test loss')
axes[1].set(title='Future loss: LSTM vs MLP', xlabel='Epoch', ylabel='Standardized-return MSE')
axes[1].legend()

plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharey=True)

axes[0].plot(dates_train, y_train_actual * 10_000, label='Actual', linewidth=1.0, alpha=.8)
axes[0].plot(dates_train, lstm_train * 10_000, label='LSTM', linewidth=1.2)
axes[0].plot(dates_train, mlp_train * 10_000, label='MLP', linewidth=1.0, alpha=.8)
axes[0].set(title=f'{SYMBOL} training period (past 70%)', ylabel='Next-day return (bp)')
axes[0].legend()

axes[1].plot(dates_test, y_test_actual * 10_000, label='Actual', linewidth=1.0, alpha=.8)
axes[1].plot(dates_test, lstm_test * 10_000, label='LSTM', linewidth=1.2)
axes[1].plot(dates_test, mlp_test * 10_000, label='MLP', linewidth=1.0, alpha=.8)
axes[1].set(title=f'{SYMBOL} testing period (future 30%)', xlabel='Date', ylabel='Next-day return (bp)')
axes[1].legend()
plt.tight_layout(); plt.show()

def metrics(actual, predicted):
    return {
        'MSE': float(np.mean((actual - predicted) ** 2)),
        'MAE': float(np.mean(np.abs(actual - predicted))),
        'Directional accuracy': float(np.mean(np.sign(actual) == np.sign(predicted))),
    }

results = pd.DataFrame({
    'LSTM train': metrics(y_train_actual, lstm_train),
    'LSTM test': metrics(y_test_actual, lstm_test),
    'MLP train': metrics(y_train_actual, mlp_train),
    'MLP test': metrics(y_test_actual, mlp_test),
}).T
display(results.style.format({'MSE': '{:.8f}', 'MAE': '{:.6f}', 'Directional accuracy': '{:.2%}'}))

lstm_mse, mlp_mse = results.loc['LSTM test', 'MSE'], results.loc['MLP test', 'MSE']
if lstm_mse < mlp_mse:
    improvement = (mlp_mse - lstm_mse) / mlp_mse
    print(f'VERDICT: Yes—the LSTM beat the MLP on future MSE by {improvement:.1%}.')
else:
    difference = (lstm_mse - mlp_mse) / mlp_mse
    print(f'VERDICT: No—the LSTM future MSE was {difference:.1%} worse than the MLP.')

## 3. Daily versus weekly rebalancing after 0.5% commission

A weekly trading model should predict a weekly target—not reuse a next-day forecast. We therefore train a second copy of the same LSTM to predict the forward **five-session return**. The daily strategy trades from the next-day forecast; the weekly strategy trades once every five sessions from the five-session forecast. Both hold ABUK when the prediction is positive and otherwise hold cash.

In [ ]:
# Five-session target known at each feature date.
y5_full = np.full(feed.n_days, np.nan)
y5_full[:-5] = feed.close[5:, asset] / feed.close[:-5, asset] - 1.0
y5 = y5_full[valid].astype('float32')
y5_train_raw, y5_test_raw = y5[:split], y5[split:]

Xtr_week, y5tr_raw, dates_week_train = make_sequences(
    X_train_scaled, y5_train_raw, dates_train_raw, SEQ_LEN
)
Xte_week, y5te_raw, dates_week_test = make_sequences(
    X_test_scaled, y5_test_raw, dates_test_raw, SEQ_LEN
)
train_ok, test_ok = np.isfinite(y5tr_raw), np.isfinite(y5te_raw)
Xtr_week, y5tr_raw, dates_week_train = Xtr_week[train_ok], y5tr_raw[train_ok], dates_week_train[train_ok]
Xte_week, y5te_raw, dates_week_test = Xte_week[test_ok], y5te_raw[test_ok], dates_week_test[test_ok]

y5_mean, y5_std = float(y5tr_raw.mean()), float(y5tr_raw.std())
y5_std = y5_std if y5_std >= 1e-8 else 1.0
y5tr = ((y5tr_raw - y5_mean) / y5_std).astype('float32')
y5te = ((y5te_raw - y5_mean) / y5_std).astype('float32')

torch.manual_seed(7)
weekly_lstm = LSTMRegressor(n_features=Xtr_week.shape[2], hidden=32)
weekly_history = train_model(
    weekly_lstm, Xtr_week, y5tr, Xte_week, y5te, epochs=EPOCHS, lr=LR
)
weekly_predictions = predict(weekly_lstm, Xte_week) * y5_std + y5_mean

print(f'Weekly model training sequences: {len(Xtr_week):,}')
print(f'Weekly model testing sequences:  {len(Xte_week):,}')
print('The test is still strictly after the training period:', dates_week_train[-1] < dates_week_test[0])

In [ ]:
COMMISSION = 0.005
STARTING_CAPITAL = 1_000.0

def strategy_returns(predictions, realized_returns, step):
    pick = np.arange(0, len(predictions), step)
    positions = (predictions[pick] > 0).astype(float)
    previous = np.r_[0.0, positions[:-1]]
    turnover = np.abs(positions - previous)  # cash is the other portfolio weight
    net_returns = positions * realized_returns[pick] - COMMISSION * turnover
    return pick, positions, turnover, net_returns

# Use the same 330-session future interval for both strategies.
common_days = (len(y5te_raw) // 5) * 5
daily_pick, daily_position, daily_turnover, daily_net = strategy_returns(
    lstm_test[:common_days], y_test_actual[:common_days], step=1
)
weekly_pick, weekly_position, weekly_turnover, weekly_net = strategy_returns(
    weekly_predictions[:common_days], y5te_raw[:common_days], step=5
)

daily_equity = STARTING_CAPITAL * np.cumprod(1.0 + daily_net)
weekly_equity = STARTING_CAPITAL * np.cumprod(1.0 + weekly_net)

def backtest_metrics(net_returns, equity, turnover, periods_per_year):
    volatility = net_returns.std(ddof=1)
    sharpe = np.sqrt(periods_per_year) * net_returns.mean() / volatility if volatility > 0 else np.nan
    running_peak = np.maximum.accumulate(np.r_[STARTING_CAPITAL, equity])
    curve = np.r_[STARTING_CAPITAL, equity]
    drawdown = curve / running_peak - 1.0
    return {
        'Ending EGP': equity[-1],
        'Total return': equity[-1] / STARTING_CAPITAL - 1.0,
        'Sharpe': sharpe,
        'Max drawdown': drawdown.min(),
        'Trades': int(np.count_nonzero(turnover)),
        'Total turnover': turnover.sum(),
    }

comparison = pd.DataFrame({
    'Daily rebalance': backtest_metrics(daily_net, daily_equity, daily_turnover, 252),
    'Weekly rebalance': backtest_metrics(weekly_net, weekly_equity, weekly_turnover, 252 / 5),
}).T

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(dates_test[:common_days], daily_equity, label='Daily rebalance', linewidth=1.5)
ax.plot(dates_week_test[weekly_pick], weekly_equity, label='Weekly rebalance', linewidth=2.0)
ax.axhline(STARTING_CAPITAL, color='gray', linestyle='--', linewidth=1)
ax.set(title=f'{SYMBOL} LSTM strategy after 0.5% commission', xlabel='Date', ylabel='Portfolio value (EGP)')
ax.legend(); plt.tight_layout(); plt.show()

display(comparison.style.format({
    'Ending EGP': '{:,.2f}', 'Total return': '{:.2%}', 'Sharpe': '{:.3f}',
    'Max drawdown': '{:.2%}', 'Trades': '{:.0f}', 'Total turnover': '{:.1f}',
}))

winner = comparison['Ending EGP'].idxmax()
difference = comparison.loc['Weekly rebalance', 'Ending EGP'] - comparison.loc['Daily rebalance', 'Ending EGP']
print(f'REBALANCING VERDICT: {winner} finished with more money.')
print(f'Weekly minus daily ending value: {difference:+.2f} EGP.')

## 4. Honest search for a better rebalance horizon

Trying many settings on the final test would make the winner unreliable. We now create three chronological periods: **60% train**, **20% validation**, and **20% final test**. Both models receive the same 10-day information: the LSTM sees a `(10, 9)` sequence and the MLP sees the identical sequence flattened to 90 inputs.

We compare 5-, 10-, and 20-session targets on validation data. One common horizon is selected using the average validation ending value of both architectures, then the LSTM and MLP are compared once on the untouched final period.

In [ ]:
HORIZONS = [5, 10, 20]
MODEL_LOOKBACK = 10
TRAIN_END = int(0.60 * len(X))
VALID_END = int(0.80 * len(X))

# This experiment gets its own training-only feature normalization.
search_x_mean = X[:TRAIN_END].mean(axis=0)
search_x_std = X[:TRAIN_END].std(axis=0)
search_x_std[search_x_std < 1e-8] = 1.0
X_search = ((X - search_x_mean) / search_x_std).astype('float32')

def build_horizon_data(horizon):
    target_full = np.full(feed.n_days, np.nan)
    target_full[:-horizon] = feed.close[horizon:, asset] / feed.close[:-horizon, asset] - 1.0
    target = target_full[valid].astype('float32')
    end_indices = np.arange(MODEL_LOOKBACK - 1, len(X))
    sequences = np.array([X_search[i-MODEL_LOOKBACK+1:i+1] for i in end_indices], dtype='float32')
    targets = target[end_indices]
    sequence_dates = dates[end_indices]

    train_mask = (end_indices + horizon < TRAIN_END) & np.isfinite(targets)
    valid_mask = (end_indices >= TRAIN_END) & (end_indices + horizon < VALID_END) & np.isfinite(targets)
    test_mask = (end_indices >= VALID_END) & np.isfinite(targets)
    return {
        'train': (sequences[train_mask], targets[train_mask], sequence_dates[train_mask]),
        'validation': (sequences[valid_mask], targets[valid_mask], sequence_dates[valid_mask]),
        'test': (sequences[test_mask], targets[test_mask], sequence_dates[test_mask]),
    }

search_runs = {}
for horizon in HORIZONS:
    parts = build_horizon_data(horizon)
    X_train_h, y_train_h_raw, _ = parts['train']
    X_valid_h, y_valid_h_raw, _ = parts['validation']
    X_test_h, y_test_h_raw, _ = parts['test']
    target_mean, target_std = float(y_train_h_raw.mean()), float(y_train_h_raw.std())
    target_std = target_std if target_std >= 1e-8 else 1.0
    y_train_h = ((y_train_h_raw - target_mean) / target_std).astype('float32')
    y_valid_h = ((y_valid_h_raw - target_mean) / target_std).astype('float32')

    torch.manual_seed(7)
    candidate_lstm = LSTMRegressor(X_train_h.shape[2], hidden=32)
    lstm_h_history = train_model(
        candidate_lstm, X_train_h, y_train_h, X_valid_h, y_valid_h, epochs=EPOCHS, lr=LR
    )

    X_train_flat = X_train_h.reshape(len(X_train_h), -1)
    X_valid_flat = X_valid_h.reshape(len(X_valid_h), -1)
    X_test_flat = X_test_h.reshape(len(X_test_h), -1)
    torch.manual_seed(7)
    candidate_mlp = DeepMLP(X_train_flat.shape[1], hidden=32, n_hidden_layers=2)
    mlp_h_history = train_model(
        candidate_mlp, X_train_flat, y_train_h, X_valid_flat, y_valid_h, epochs=EPOCHS, lr=LR
    )

    search_runs[horizon] = {
        'parts': parts,
        'LSTM validation': predict(candidate_lstm, X_valid_h) * target_std + target_mean,
        'MLP validation': predict(candidate_mlp, X_valid_flat) * target_std + target_mean,
        'LSTM test': predict(candidate_lstm, X_test_h) * target_std + target_mean,
        'MLP test': predict(candidate_mlp, X_test_flat) * target_std + target_mean,
        'LSTM history': lstm_h_history,
        'MLP history': mlp_h_history,
    }

print('Finished training matched LSTM and MLP models for horizons:', HORIZONS)
print(f'Final test begins at index {VALID_END} and was not used to choose the horizon.')

In [ ]:
ROUND_TRIP_HURDLE = 2 * COMMISSION  # 1%: one buy plus one later sell

def hurdle_backtest(predictions, realized, horizon):
    chosen = np.arange(0, len(predictions), horizon)
    position = 0.0
    positions, turnovers, net_returns = [], [], []
    for i in chosen:
        if predictions[i] > ROUND_TRIP_HURDLE:
            new_position = 1.0
        elif predictions[i] < 0.0:
            new_position = 0.0
        else:
            new_position = position
        turnover = abs(new_position - position)
        net_returns.append(new_position * realized[i] - COMMISSION * turnover)
        positions.append(new_position); turnovers.append(turnover)
        position = new_position
    net_returns = np.asarray(net_returns)
    turnovers = np.asarray(turnovers)
    equity = STARTING_CAPITAL * np.cumprod(1.0 + net_returns)
    stats = backtest_metrics(net_returns, equity, turnovers, 252 / horizon)
    return chosen, equity, stats

validation_rows = []
for horizon, run in search_runs.items():
    _, validation_actual, _ = run['parts']['validation']
    for model_name in ['LSTM', 'MLP']:
        _, _, stats = hurdle_backtest(run[f'{model_name} validation'], validation_actual, horizon)
        validation_rows.append({'Horizon': horizon, 'Model': model_name, **stats})
validation_results = pd.DataFrame(validation_rows)
validation_pivot = validation_results.pivot(index='Horizon', columns='Model', values='Ending EGP')
validation_pivot['Average'] = validation_pivot[['LSTM', 'MLP']].mean(axis=1)
selected_horizon = int(validation_pivot['Average'].idxmax())

print('VALIDATION ONLY — used to choose one common horizon')
display(validation_pivot.style.format('{:,.2f}').highlight_max(subset=['Average'], color='#b7e4c7'))
print(f'Selected horizon: {selected_horizon} sessions')

selected = search_runs[selected_horizon]
_, final_actual, final_dates = selected['parts']['test']
final_rows, final_curves = [], {}
for model_name in ['LSTM', 'MLP']:
    predictions = selected[f'{model_name} test']
    picked, equity, stats = hurdle_backtest(predictions, final_actual, selected_horizon)
    prediction_mse = float(np.mean((final_actual - predictions) ** 2))
    final_rows.append({'Model': model_name, 'Prediction MSE': prediction_mse, **stats})
    final_curves[model_name] = (final_dates[picked], equity)
final_results = pd.DataFrame(final_rows).set_index('Model')

fig, ax = plt.subplots(figsize=(13, 5))
for model_name, (curve_dates, curve) in final_curves.items():
    ax.plot(curve_dates, curve, label=model_name, linewidth=2)
ax.axhline(STARTING_CAPITAL, color='gray', linestyle='--', linewidth=1)
ax.set(
    title=f'Untouched final test — {selected_horizon}-session rebalance with 1% hurdle',
    xlabel='Date', ylabel='Portfolio value (EGP)'
)
ax.legend(); plt.tight_layout(); plt.show()

print('FINAL TEST — not used for model or horizon selection')
display(final_results.style.format({
    'Prediction MSE': '{:.8f}', 'Ending EGP': '{:,.2f}', 'Total return': '{:.2%}',
    'Sharpe': '{:.3f}', 'Max drawdown': '{:.2%}', 'Trades': '{:.0f}', 'Total turnover': '{:.1f}',
}))

portfolio_winner = final_results['Ending EGP'].idxmax()
prediction_winner = final_results['Prediction MSE'].idxmin()
print(f'PORTFOLIO WINNER: {portfolio_winner}')
print(f'PREDICTION-MSE WINNER: {prediction_winner}')
if portfolio_winner == 'LSTM' and prediction_winner == 'LSTM':
    print('The LSTM beat the MLP both as a predictor and as a net trading strategy on the final period.')
elif portfolio_winner == 'LSTM':
    print('The LSTM trading rule finished higher, but its raw prediction MSE did not beat the MLP.')
else:
    print('The honest final test did not show an LSTM advantage over the matched MLP.')

## Takeaway

Unlike the smooth toy wave, daily stock returns are noisy. An LSTM can remember the recent feature sequence, but extra memory only matters if that history contains a stable signal. Judge it on the future 30%, not on how closely it fits the training period.